In [4]:
# ============================================================
# Create X/Y coordinate predictor COGs from a template raster
# ============================================================

import os
import numpy as np
from osgeo import gdal

# ---------------- USER SETTINGS ----------------

template_raster = r"G:\Covariates_to_model\Topo_dtm_ch.tif"
out_dir = r"G:\Geology_rasters_2026\Coordinates"

out_x_cog = os.path.join(out_dir, "coord_X.tif")
out_y_cog = os.path.join(out_dir, "coord_Y.tif")

tmp_x = os.path.join(out_dir, "_tmp_coord_X.tif")
tmp_y = os.path.join(out_dir, "_tmp_coord_Y.tif")

coord_nodata = -99999.0
block_rows = 1024

# ---------------- SETUP ----------------

os.makedirs(out_dir, exist_ok=True)
gdal.UseExceptions()

src = gdal.Open(template_raster, gdal.GA_ReadOnly)
if src is None:
    raise RuntimeError(f"Could not open template raster: {template_raster}")

cols = src.RasterXSize
rows = src.RasterYSize
gt = src.GetGeoTransform()
proj = src.GetProjection()

src_band = src.GetRasterBand(1)
src_nodata = src_band.GetNoDataValue()

print("Template raster:")
print(f"  Path: {template_raster}")
print(f"  Columns: {cols}")
print(f"  Rows: {rows}")
print(f"  GeoTransform: {gt}")
print(f"  NoData: {src_nodata}")

# ---------------- CREATE TEMP GEOTIFFS ----------------

driver = gdal.GetDriverByName("GTiff")

creation_options = [
    "TILED=YES",
    "COMPRESS=DEFLATE",
    "PREDICTOR=3",
    "BIGTIFF=YES",
    "BLOCKXSIZE=512",
    "BLOCKYSIZE=512"
]

for f in [tmp_x, tmp_y, out_x_cog, out_y_cog]:
    if os.path.exists(f):
        os.remove(f)

dst_x = driver.Create(tmp_x, cols, rows, 1, gdal.GDT_Float32, creation_options)
dst_y = driver.Create(tmp_y, cols, rows, 1, gdal.GDT_Float32, creation_options)

for dst in [dst_x, dst_y]:
    dst.SetGeoTransform(gt)
    dst.SetProjection(proj)
    dst.GetRasterBand(1).SetNoDataValue(coord_nodata)

# ---------------- WRITE BLOCKS ----------------

for row0 in range(0, rows, block_rows):
    nrows = min(block_rows, rows - row0)

    src_arr = src_band.ReadAsArray(0, row0, cols, nrows)

    if src_nodata is None:
        valid = np.isfinite(src_arr)
    else:
        valid = (src_arr != src_nodata) & np.isfinite(src_arr)

    r = np.arange(row0, row0 + nrows, dtype=np.float64) + 0.5
    c = np.arange(0, cols, dtype=np.float64) + 0.5

    cc, rr = np.meshgrid(c, r)

    x_arr = gt[0] + cc * gt[1] + rr * gt[2]
    y_arr = gt[3] + cc * gt[4] + rr * gt[5]

    x_arr = x_arr.astype(np.float32)
    y_arr = y_arr.astype(np.float32)

    x_arr[~valid] = coord_nodata
    y_arr[~valid] = coord_nodata

    dst_x.GetRasterBand(1).WriteArray(x_arr, 0, row0)
    dst_y.GetRasterBand(1).WriteArray(y_arr, 0, row0)

    print(f"Processed rows {row0 + 1} to {row0 + nrows} / {rows}")

dst_x.FlushCache()
dst_y.FlushCache()

dst_x = None
dst_y = None
src = None

# ---------------- CONVERT TO COG ----------------

cog_options = [
    "COMPRESS=DEFLATE",
    "PREDICTOR=3",
    "BIGTIFF=YES",
    "BLOCKSIZE=512",
    "OVERVIEWS=AUTO"
]

print("Creating X COG...")
gdal.Translate(out_x_cog, tmp_x, format="COG", creationOptions=cog_options)

print("Creating Y COG...")
gdal.Translate(out_y_cog, tmp_y, format="COG", creationOptions=cog_options)

# Optional cleanup
os.remove(tmp_x)
os.remove(tmp_y)

print("\nDone.")
print(f"X coordinate COG: {out_x_cog}")
print(f"Y coordinate COG: {out_y_cog}")

Template raster:
  Path: G:\Covariates_to_model\Topo_dtm_ch.tif
  Columns: 135236
  Rows: 162232
  GeoTransform: (-154809.95275266352, 10.0, 0.0, 8020664.6625600215, 0.0, -10.0)
  NoData: -3.4028230607370965e+38
Processed rows 1 to 1024 / 162232
Processed rows 1025 to 2048 / 162232
Processed rows 2049 to 3072 / 162232
Processed rows 3073 to 4096 / 162232
Processed rows 4097 to 5120 / 162232
Processed rows 5121 to 6144 / 162232
Processed rows 6145 to 7168 / 162232
Processed rows 7169 to 8192 / 162232
Processed rows 8193 to 9216 / 162232
Processed rows 9217 to 10240 / 162232
Processed rows 10241 to 11264 / 162232
Processed rows 11265 to 12288 / 162232
Processed rows 12289 to 13312 / 162232
Processed rows 13313 to 14336 / 162232
Processed rows 14337 to 15360 / 162232
Processed rows 15361 to 16384 / 162232
Processed rows 16385 to 17408 / 162232
Processed rows 17409 to 18432 / 162232
Processed rows 18433 to 19456 / 162232
Processed rows 19457 to 20480 / 162232
Processed rows 20481 to 21504 

<class 'PermissionError'>: [WinError 32] The process cannot access the file because it is being used by another process: 'G:\\Geology_rasters_2026\\Coordinates\\_tmp_coord_Y.tif'